# RegresiLab Unpad — Launcher Google Colab
**Aplikasi Streamlit dijalankan di runtime Colab melalui tunnel ngrok.** Anda memerlukan [ngrok authtoken](https://dashboard.ngrok.com/get-started/your-authtoken) milik sendiri. Hindari data sensitif karena tunnel publik. Sesi berhenti saat runtime Colab dihentikan.

In [ ]:
from google.colab import files
uploaded = files.upload()
assert len(uploaded)==1 and next(iter(uploaded)).lower().endswith(".zip"), "Upload ZIP paket RegresiLab Unpad"
import zipfile, os
archive=next(iter(uploaded))
with zipfile.ZipFile(archive) as f:
    if any(p.startswith("/") or ".." in p.split("/") for p in f.namelist()): raise ValueError("ZIP tidak aman")
    f.extractall("/content/regresilab")
print("Berkas diekstrak")

In [ ]:
import pathlib
root=next(p.parent for p in pathlib.Path("/content/regresilab").rglob("app.py") if p.parent.joinpath("requirements.txt").exists())
print("Project:",root)
%pip -q install -r {str(root / "requirements.txt")} pyngrok

In [ ]:
from getpass import getpass
from pyngrok import ngrok
import subprocess, time
ngrok.set_auth_token(getpass("Tempel authtoken ngrok: "))
proc=subprocess.Popen(["streamlit","run",str(root / "app.py"),"--server.port","8501","--server.headless","true"],cwd=str(root),stdout=open("/content/streamlit.log","w"),stderr=subprocess.STDOUT)
time.sleep(6)
if proc.poll() is not None: print(open("/content/streamlit.log").read());raise RuntimeError("Streamlit gagal start")
tunnel=ngrok.connect(8501, "http")
print("Buka tautan publik sementara:",tunnel.public_url)

In [ ]:
# Jalankan untuk menutup layanan
ngrok.kill()
proc.terminate()
print("Sesi dihentikan")